In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
eeg_eye_state = fetch_ucirepo(id=264) 
  
# data (as pandas dataframes) 
X = eeg_eye_state.data.features 
y = eeg_eye_state.data.targets 
  
# metadata 
print(eeg_eye_state.metadata) 
  
# variable information 
print(eeg_eye_state.variables) 

{'uci_id': 264, 'name': 'EEG Eye State', 'repository_url': 'https://archive.ics.uci.edu/dataset/264/eeg+eye+state', 'data_url': 'https://archive.ics.uci.edu/static/public/264/data.csv', 'abstract': 'The data set consists of 14 EEG values and a value indicating the eye state.', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 14980, 'num_features': 14, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['eyeDetection'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2013, 'last_updated': 'Thu Mar 21 2024', 'dataset_doi': '10.24432/C57G7J', 'creators': ['Oliver Roesler'], 'intro_paper': None, 'additional_info': {'summary': "All data is from one continuous EEG measurement with the Emotiv EEG Neuroheadset. The duration of the measurement was 117 seconds. The eye state was detected via a camera during the EEG measuremen

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [3]:
features = X
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.target = y
df.head()

/var/folders/4y/0qtvj2hs6r9_dvmt7n7fbdk40000gn/T/ipykernel_12966/2312714648.py:5: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.target = y


,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4329.23,4009.23,4289.23,4148.21,4350.26,4586.15,4096.92,4641.03,4222.05,4238.46,4211.28,4280.51,4635.90,4393.85
1,4324.62,4004.62,4293.85,4148.72,4342.05,4586.67,4097.44,4638.97,4210.77,4226.67,4207.69,4279.49,4632.82,4384.10
2,4327.69,4006.67,4295.38,4156.41,4336.92,4583.59,4096.92,4630.26,4207.69,4222.05,4206.67,4282.05,4628.72,4389.23
3,4328.72,4011.79,4296.41,4155.90,4343.59,4582.56,4097.44,4630.77,4217.44,4235.38,4210.77,4287.69,4632.31,4396.41
4,4326.15,4011.79,4292.31,4151.28,4347.69,4586.67,4095.90,4627.69,4210.77,4244.10,4212.82,4288.21,4632.82,4398.46


In [4]:
fs = 128
signal = X['AF3'].values

N = len(signal)

fft_vals = np.fft.fft(signal)
freqs = np.fft.fftfreq(N, d=1/fs)

alpha_mask = (freqs >= 8) & (freqs <= 12)

alpha_fft = np.zeros_like(fft_vals)
alpha_fft[alpha_mask] = fft_vals[alpha_mask]

alpha_signal = np.fft.ifft(alpha_fft).real

In [5]:
alpha_signal

array([  8.20296219,  -5.9581875 , -19.09351972, ...,  24.49123646,
        25.42309617,  19.61698298], shape=(14980,))

In [6]:
alpha_data = []

In [7]:
for i in X.columns:
    signal = X[i].values
    N = len(signal)

    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1/fs)

    alpha_mask = (freqs >= 8) & (freqs <= 12)

    alpha_fft = np.zeros_like(fft_vals)
    alpha_fft[alpha_mask] = fft_vals[alpha_mask]

    alpha_signal = np.fft.ifft(alpha_fft).real

    alpha_data.append(alpha_signal)

In [8]:
np.array(alpha_data)

array([[  8.20296219,  -5.9581875 , -19.09351972, ...,  24.49123646,
         25.42309617,  19.61698298],
       [  1.27203488,   1.62759711,   1.54702377, ...,  -1.17097123,
         -0.33302025,   0.55843418],
       [  0.76205752,   2.12766106,   2.96011083, ...,  -2.99150416,
         -2.16238682,  -0.79822496],
       ...,
       [  0.37532698,   1.57883162,   2.3290318 , ...,  -2.78803729,
         -2.13668558,  -0.98057503],
       [  4.68881929,  -2.11134123,  -8.77635337, ...,  11.32616026,
         12.18013238,   9.85753976],
       [ 13.91763026,   9.4353329 , -10.07059665, ..., -94.67531978,
        -45.76069268,  -5.23395047]], shape=(14, 14980))

In [9]:
alpha_data = np.array(alpha_data)
alpha_data = np.transpose(alpha_data)

features = alpha_data
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.head(10)

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,8.202962,1.272035,0.762058,-48.925535,-0.257961,9.701641,-43.220640,1.832916,6.799344,1.561678,1.022907,0.375327,4.688819,13.917630
1,-5.958187,1.627597,2.127661,-47.106257,-0.054831,6.401935,-41.607006,2.724367,-5.165026,2.106097,2.154695,1.578832,-2.111341,9.435333
2,-19.093520,1.547024,2.960111,-34.058761,0.082836,-4.807641,-30.038627,2.828757,-16.220515,1.858185,2.548060,2.329032,-8.776353,-10.070597
3,-27.638570,1.075849,3.063046,-13.597156,0.165365,-15.775171,-11.887809,2.069761,-23.489071,0.935383,2.171685,2.459064,-13.503745,-29.089652
4,-29.305197,0.368376,2.432290,8.572821,0.226847,-18.188576,7.779227,0.629786,-25.126026,-0.313825,1.231284,1.978620,-14.993032,-31.793349
5,-23.757828,-0.359638,1.252370,26.598151,0.301601,-7.083975,23.752280,-1.096016,-20.845067,-1.428820,0.089373,1.063865,-12.835762,-8.808503
6,-12.686151,-0.896694,-0.158763,36.209171,0.402108,16.838528,32.222400,-2.610712,-11.967176,-1.993703,-0.861071,-0.001459,-7.635195,38.321344
7,0.742329,-1.099587,-1.440040,35.888831,0.507966,46.972941,31.832364,-3.459935,-0.986568,-1.780619,-1.327103,-0.914937,-0.819648,96.692153
8,12.867229,-0.935852,-2.285962,27.126285,0.570853,72.797574,23.912289,-3.367384,9.199442,-0.831956,-1.207925,-1.446382,5.789089,145.915150
9,20.637539,-0.491150,-2.525185,13.707607,0.534015,83.466231,11.853337,-2.321126,16.146142,0.546739,-0.619263,-1.502389,10.514326,165.083473


In [10]:
noalpha_data = []

In [11]:
for i in X.columns:
    signal = X[i].values
    N = len(signal)

    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1/fs)

    noalpha_mask = (freqs <= 8) | (freqs >= 12)

    noalpha_fft = np.zeros_like(fft_vals)
    noalpha_fft[noalpha_mask] = fft_vals[noalpha_mask]

    noalpha_signal = np.fft.ifft(noalpha_fft).real

    noalpha_data.append(noalpha_signal)

In [12]:
noalpha_data = np.array(noalpha_data)
noalpha_data = np.transpose(noalpha_data)

features = noalpha_data
feature_names = X.columns

df = pd.DataFrame(features,columns = feature_names)
df.head(10)

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4321.027038,4007.957965,4288.467942,4197.135535,4350.517961,4576.448359,4140.140640,4639.197084,4215.250656,4236.898322,4210.257093,4280.134673,4631.211181,4379.932370
1,4330.578187,4002.992403,4291.722339,4195.826257,4342.104831,4580.268065,4139.047006,4636.245633,4215.935026,4224.563903,4205.535305,4277.911168,4634.931341,4374.664667
2,4346.783520,4005.122976,4292.419889,4190.468761,4336.837164,4588.397641,4126.958627,4627.431243,4223.910515,4220.191815,4204.121940,4279.720968,4637.496353,4399.300597
3,4356.358570,4010.714151,4293.346954,4169.497156,4343.424635,4598.335171,4109.327809,4628.700239,4240.929071,4234.444617,4208.598315,4285.230936,4645.813745,4425.499652
4,4355.455197,4011.421624,4289.877710,4142.707179,4347.463153,4604.858576,4088.120773,4627.060214,4235.896026,4244.413825,4211.588716,4286.231380,4647.813032,4430.253349
5,4344.787828,4004.979638,4282.847630,4126.731849,4345.338399,4594.263975,4069.577720,4618.016016,4223.405067,4234.248820,4209.650627,4279.966135,4641.045762,4398.548503
6,4332.176151,4001.926694,4280.668763,4115.580829,4343.187892,4567.781472,4057.517600,4618.510712,4224.277176,4228.663703,4201.891071,4269.741459,4632.765195,4340.138656
7,4324.897671,4007.769587,4279.900040,4107.191169,4343.592034,4536.107059,4055.347636,4618.329935,4206.626568,4232.040619,4197.227103,4267.584937,4622.869648,4283.817847
8,4313.282771,4011.705852,4278.695962,4112.363715,4344.559147,4511.302426,4067.367711,4611.577384,4178.490558,4230.571956,4203.257925,4275.296382,4621.390911,4243.824850
9,4305.512461,4011.771150,4279.445185,4128.342393,4343.565985,4499.093769,4080.966663,4611.041126,4178.213858,4228.173261,4213.439263,4279.452389,4626.925674,4228.246527


In [13]:
y = y['eyeDetection']

alpha_data_train, alpha_data_test, y_train, y_test = train_test_split(alpha_data, y, test_size=0.2, random_state=67)

In [14]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(alpha_data_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [15]:
alpha_data_train = scaler.transform(alpha_data_train)
alpha_data_test = scaler.transform(alpha_data_test)

In [16]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [18]:
xgb.fit(alpha_data_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [20]:
y_pred = xgb.predict(alpha_data_test)
from sklearn import metrics
metrics.accuracy_score(y_test, y_pred)

0.7696929238985314

In [21]:
noalpha_data_train, noalpha_data_test, y_train, y_test = train_test_split(noalpha_data, y, test_size=0.2, random_state=67)

In [23]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(noalpha_data_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [24]:
noalpha_data_train = scaler.transform(noalpha_data_train)
noalpha_data_test = scaler.transform(noalpha_data_test)

In [26]:
xgb.fit(noalpha_data_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [27]:
y_pred = xgb.predict(noalpha_data_test)
from sklearn import metrics
metrics.accuracy_score(y_test, y_pred)

0.8885180240320427